In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, recall_score, classification_report
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
df = pd.read_csv(r'C:\Users\kanna\hospital-readmission-predictor\data\cleaned_data.csv')
print(f"Shape: {df.shape}")
print("Libraries loaded successfully")

Shape: (69973, 49)
Libraries loaded successfully


In [2]:
# Encode categorical columns to numbers
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'readmitted']

print(f"Encoding {len(cat_cols)} categorical columns...")

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print("Encoding complete")
print(f"Categorical columns encoded: {cat_cols}")

Encoding 33 categorical columns...
Encoding complete
Categorical columns encoded: ['race', 'gender', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


In [3]:
# Split data into features and target
X = df.drop(columns=['target', 'readmitted', 'patient_nbr', 'encounter_id'])
y = df['target']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTarget distribution in training set:")
print(y_train.value_counts())
print(f"\nTarget distribution in test set:")
print(y_test.value_counts())

Training set: (55978, 45)
Test set: (13995, 45)

Target distribution in training set:
target
0    50956
1     5022
Name: count, dtype: int64

Target distribution in test set:
target
0    12740
1     1255
Name: count, dtype: int64


In [4]:
# Model 1: Logistic Regression (baseline)
print("Training Logistic Regression...")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

lr_auc = roc_auc_score(y_test, lr_prob)
lr_recall = recall_score(y_test, lr_pred)

print(f"Logistic Regression Results:")
print(f"  ROC-AUC: {lr_auc:.3f}")
print(f"  Recall:  {lr_recall:.3f}")

Training Logistic Regression...
Logistic Regression Results:
  ROC-AUC: 0.624
  Recall:  0.516


In [5]:
# Model 2: Random Forest
print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_auc = roc_auc_score(y_test, rf_prob)
rf_recall = recall_score(y_test, rf_pred)

print(f"Random Forest Results:")
print(f"  ROC-AUC: {rf_auc:.3f}")
print(f"  Recall:  {rf_recall:.3f}")

Training Random Forest...
Random Forest Results:
  ROC-AUC: 0.625
  Recall:  0.002


In [6]:
# Model 3: XGBoost
print("Training XGBoost...")

# Calculate scale_pos_weight to handle class imbalance
scale = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale:.1f}")

xgb = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='auc',
    verbosity=0
)
xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)[:, 1]

xgb_auc = roc_auc_score(y_test, xgb_prob)
xgb_recall = recall_score(y_test, xgb_pred)

print(f"\nXGBoost Results:")
print(f"  ROC-AUC: {xgb_auc:.3f}")
print(f"  Recall:  {xgb_recall:.3f}")

Training XGBoost...
scale_pos_weight: 10.1

XGBoost Results:
  ROC-AUC: 0.609
  Recall:  0.398


In [7]:
# Model comparison
print("=== Model Comparison ===")
print(f"{'Model':<25} {'ROC-AUC':<12} {'Recall':<12}")
print("-" * 49)
print(f"{'Logistic Regression':<25} {lr_auc:<12.3f} {lr_recall:<12.3f}")
print(f"{'Random Forest':<25} {rf_auc:<12.3f} {rf_recall:<12.3f}")
print(f"{'XGBoost':<25} {xgb_auc:<12.3f} {xgb_recall:<12.3f}")

print("\n=== Winner ===")
best_auc = max(lr_auc, rf_auc, xgb_auc)
if best_auc == lr_auc:
    print("Logistic Regression wins on ROC-AUC")
elif best_auc == rf_auc:
    print("Random Forest wins on ROC-AUC")
else:
    print("XGBoost wins on ROC-AUC")

=== Model Comparison ===
Model                     ROC-AUC      Recall      
-------------------------------------------------
Logistic Regression       0.624        0.516       
Random Forest             0.625        0.002       
XGBoost                   0.609        0.398       

=== Winner ===
Random Forest wins on ROC-AUC


In [8]:
# Save the best model
import pickle

with open(r'C:\Users\kanna\hospital-readmission-predictor\data\best_model.pkl', 'wb') as f:
    pickle.dump(lr, f)

# Save feature names
feature_names = X_train.columns.tolist()
with open(r'C:\Users\kanna\hospital-readmission-predictor\data\feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print("Best model (Logistic Regression) saved!")
print(f"Features: {len(feature_names)}")

Best model (Logistic Regression) saved!
Features: 45
